# E30 --- quanto custa contar os pares

O capítulo do dia em que tudo cai mede a dependência contando DIAS: quantos dias as duas pernas
rompem juntas. Esta medição faz a outra pergunta --- os dois mercados andam juntos? --- olhando o
**par de dias** em vez do dia, e cobra o preço: sobre os mesmos dias que o capítulo conta há
dezenove milhões de pares, e a economia de olhar só uma parte deles tem uma barra que se prevê e se
mede. Os dias usados são os mesmos do capítulo, o mesmo corte e a mesma janela.


In [1]:
import json
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, dependencia, graficos, pares, volatilidade

RAIZ = Path.cwd()
SERIE_A, SERIE_B = "sp500.csv", "ibov.csv"
JANELA, CAUDA = 252, 0.05
FRACOES = (1.0, 0.5, 0.1, 0.01)
MUNDOS = 40
SEMENTE = pares.SEMENTE_PADRAO

retornos_a = volatilidade.retornos_log(dados.carregar_serie(SERIE_A))
retornos_b = volatilidade.retornos_log(dados.carregar_serie(SERIE_B))
rompe_a = dependencia.rompimentos(retornos_a, JANELA, CAUDA)
rompe_b = dependencia.rompimentos(retornos_b, JANELA, CAUDA)
dias = rompe_a.index.intersection(rompe_b.index)
a, b = retornos_a.loc[dias], retornos_b.loc[dias]
print("frevolab %s | %s e %s | %d dias (os mesmos do capitulo)" % (frevolab.VERSAO, SERIE_A, SERIE_B, dias.size))


frevolab 0.1.0 | sp500.csv e ibov.csv | 6204 dias (os mesmos do capitulo)


In [2]:
# A concordancia exata, o numero de pares que ela resume, e o preco do gesto caro.
sorteio = np.random.default_rng(SEMENTE)
t0 = time.time()
exato = pares.concordancia(a, b)
tempo_exato = time.time() - t0
momentos = pares.momentos_do_kernel(a, b, rng=sorteio)
C = exato["pares"]
print("concordancia exata: %.4f | dias %d | pares %d | calculada em %.3f s"
      % (exato["tau"], exato["dias"], C, tempo_exato))
print("sigma^2 = 1 - tau^2 = %.4f | sigma_1^2 (dois pares que dividem um dia) = %.4f"
      % (momentos["sigma_2"], momentos["sigma_1_2"]))

# O gesto caro, medido no pedaco: enumerar um centesimo dos pares e cronometrar.
quanto = max(1, C // 100)
t0 = time.time()
i = sorteio.integers(0, exato["dias"], size=quanto)
j = sorteio.integers(0, exato["dias"], size=quanto)
kernel = np.sign((a.to_numpy()[i] - a.to_numpy()[j]) * (b.to_numpy()[i] - b.to_numpy()[j]))
tempo_centesimo = time.time() - t0
print("enumerar %d pares custou %.2f s; enumerar todos custaria, no mesmo passo, cerca de %.0f s"
      % (quanto, tempo_centesimo, 100 * tempo_centesimo))


concordancia exata: 0.3528 | dias 6204 | pares 19241706 | calculada em 0.002 s
sigma^2 = 1 - tau^2 = 0.8755 | sigma_1^2 (dois pares que dividem um dia) = 0.0989
enumerar 192417 pares custou 0.00 s; enumerar todos custaria, no mesmo passo, cerca de 0 s


In [3]:
# As duas barras: a do sorteio (sigma/sqrt(m)) e a do dado (Hoeffding).
linhas = pares.varredura(a, b, fracoes=FRACOES, mundos=MUNDOS, rng=np.random.default_rng(SEMENTE + 1))
quadro = pd.DataFrame(linhas)
print(quadro.to_string(index=False, float_format=lambda v: "%.5f" % v))
do_dado = linhas[0]["desvio_do_dado"]
print()
print("barra do DADO (a que o mercado impoe sozinho): %.5f" % do_dado)
for linha in linhas:
    print("  %6.2f%% dos pares (%9d): barra do sorteio %.5f -> %.0f%% da barra do dado"
          % (100 * linha["fracao"], linha["pares"], linha["desvio_previsto"],
             100 * linha["desvio_previsto"] / do_dado))
# O que a economia custa em variancia: a barra do sorteio no centesimo, contra a do dado.
centesimo = [l for l in linhas if abs(l["fracao"] - 0.01) < 1e-9][0]
custo = pares.variancia_amostral(momentos["sigma_2"], centesimo["pares"]) / pares.variancia_do_completo(
    exato["dias"], momentos["sigma_2"], momentos["sigma_1_2"])
print("com um centesimo dos pares, o sorteio custa %.1f%% da variancia que o dado ja impoe" % (100 * custo))


 fracao    pares  tau_medio  desvio_medido  desvio_previsto  desvio_do_dado   exato
1.00000 19241706    0.35284        0.00019          0.00021         0.00812 0.35282
0.50000  9620853    0.35274        0.00028          0.00030         0.00812 0.35282
0.10000  1924171    0.35290        0.00069          0.00067         0.00812 0.35282
0.01000   192417    0.35321        0.00191          0.00213         0.00812 0.35282

barra do DADO (a que o mercado impoe sozinho): 0.00812
  100.00% dos pares ( 19241706): barra do sorteio 0.00021 -> 3% da barra do dado
   50.00% dos pares (  9620853): barra do sorteio 0.00030 -> 4% da barra do dado
   10.00% dos pares (  1924171): barra do sorteio 0.00067 -> 8% da barra do dado
    1.00% dos pares (   192417): barra do sorteio 0.00213 -> 26% da barra do dado
com um centesimo dos pares, o sorteio custa 7.1% da variancia que o dado ja impoe


In [4]:
# Figura 1: a barra do sorteio contra a fracao de pares, medida e prevista, e a barra do dado.
fig, eixo = plt.subplots(figsize=(7.6, 4.0))
fracoes = np.array([l["fracao"] for l in linhas])
eixo.loglog(100 * fracoes, [l["desvio_medido"] for l in linhas], marker="o", lw=1.8,
            color="#1f4e79", label="barra medida (%d sorteios)" % MUNDOS)
eixo.loglog(100 * fracoes, [l["desvio_previsto"] for l in linhas], lw=1.4, ls="--",
            color="#b03a2e", label="a lei: sigma sobre raiz de m")
eixo.axhline(do_dado, color="#2e7d32", lw=1.3, ls=":",
             label="a barra do dado (Hoeffding)")
eixo.set_xlabel("pares usados, em % do total")
eixo.set_ylabel("desvio da concordancia")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, which="both", ls=":")
fig.tight_layout()
graficos.salvar(fig, "E30_pares", 1)
plt.close(fig)
print("figura gravada")


figura gravada


In [5]:
# O controle do capitulo: um par sorteado independente, onde a concordancia tem de dar zero.
sorteio = np.random.default_rng(SEMENTE + 2)
falso = pd.Series(sorteio.normal(0.0, 0.01, dias.size), index=dias)
exato_falso = pares.concordancia(a, falso)
controle = []
for fracao in (1.0, 0.01):
    m = max(1, int(round(fracao * exato_falso["pares"])))
    taus = np.array([pares.concordancia_amostrada(a, falso, m, sorteio)["tau"] for _ in range(MUNDOS)])
    controle.append({"fracao": fracao, "tau_medio": float(taus.mean()),
                     "desvio_medido": float(taus.std(ddof=1)),
                     "desvio_previsto": float(np.sqrt(pares.variancia_amostral(
                         1.0 - exato_falso["tau"] ** 2, m)))})
print("par independente: concordancia exata %.5f" % exato_falso["tau"])
print(pd.DataFrame(controle).to_string(index=False, float_format=lambda v: "%.5f" % v))


par independente: concordancia exata -0.00115
 fracao  tau_medio  desvio_medido  desvio_previsto
1.00000   -0.00114        0.00020          0.00023
0.01000   -0.00036        0.00261          0.00228


## Leitura visual das figuras

Feita nesta sessão abrindo o @@E30_pares_1.png@@ com a ponte de visão (AGENTS.md §9) --- e conferida
por instrumento, medindo os pixels de cada cor no PNG, porque a afirmação desta figura é de posição
relativa.

**O que o desenho mostra.** Três coisas: a barra medida (linha cheia, com marcadores), a lei
(tracejada) e a barra do dado (linha horizontal pontilhada, no alto). O eixo horizontal é a fração
dos pares, em porcentagem, e o vertical é o desvio da concordância, em escala logarítmica.

**E o que os pixels dizem.** A linha do dado está na fileira 45 do PNG (a 254 pixels de largura,
tracejada); as curvas azul e vermelha ocupam as fileiras 202 a 492, bem abaixo. Ou seja: a medida e a
lei quase coincidem entre si, e as duas ficam muito abaixo da linha do dado --- que é exatamente o
que a tabela diz, com 0,213 a 2,133 milésimos contra 8,122. A figura não afirma nada que os números
não afirmem.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "pares_dias": int(exato["dias"]),
    "pares_total_milhoes": round(C / 1000000.0, 1),
    "pares_tau": round(float(exato["tau"]), 4),
    "pares_sigma_um": round(float(momentos["sigma_1_2"]), 4),
    "pares_sigma_dois": round(float(momentos["sigma_2"]), 4),
    "pares_mundos": int(MUNDOS),
    "pares_semente": int(SEMENTE),
    "pares_trios": int(momentos["trios"]),
    "pares_barra_do_dado": round(1000 * float(do_dado), 3),
    "pares_custo_pct": round(100 * float(custo), 1),
    "pares_tempo_exato": round(float(tempo_exato), 3),
    "pares_tempo_centesimo": round(float(tempo_centesimo), 2),
    "pares_centesimo": int(max(1, C // 100)),
    "pares_controle_tau": round(float(exato_falso["tau"]), 4),
}
NOMES = {1.0: "todos", 0.5: "meio", 0.1: "decimo", 0.01: "centesimo"}
for linha in linhas:
    nome = NOMES[round(linha["fracao"], 2)]
    resultado["pares_barra_%s" % nome] = round(1000 * float(linha["desvio_previsto"]), 3)
    resultado["pares_medido_%s" % nome] = round(1000 * float(linha["desvio_medido"]), 3)
caminho = Path("lab/resultados/E30_pares.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E30_pares.json gravado | 22 grandezas
